# Что вызывает антибиотикорезистентность?

Задача: проанализировать данные секвенации(?) штамма E. coli, который устойчив к антибиотику ампициллину. Локализовать мутации, ответственные за эту резистентность. Найти мутировавшие гены, чтобы обнаружить механизм резистентности, который работает в данном случае. Сделать рекоммендации альтернативных антибиотиков.

In [ ]:
# Готовим окружение: устанавливаем мамбу и нужные инструменты
!conda install -c conda-forge mamba -y
!mamba install -c bioconda seqkit fastqc trimmomatic bwa samtools varscan snpeff -y

## Шаг 1: загрузка данных.

Создаём директорию для файлов.

Загружаем референсный геном E.coli K-12 (MG1655) и paired-end риды резистентного штамма.

In [ ]:
!mkdir raw_data

# референсный геном (FASTA) и аннотация (GFF)
!wget -P raw_data https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/005/845/GCF_000005845.2_ASM584v2/GCF_000005845.2_ASM584v2_genomic.fna.gz
!wget -P raw_data https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/005/845/GCF_000005845.2_ASM584v2/GCF_000005845.2_ASM584v2_genomic.gff.gz

# риды с figshare:
# !wget -P raw_data https://figshare.com/ndownloader/files/23769689
# !wget -P raw_data https://figshare.com/ndownloader/files/23769692

# wget скачивал файлы весом 0б, поэтому скачала руками и положила в директорию.

## Шаг 2: ручная проверка сырых данных

In [ ]:
# на этом моменте устала писать восклицательные знаки
# загуглила что можно просто в начале ячейки указать магическую команду выбора интерпретатора

In [26]:
%%bash
cd raw_data

# смотрим первые 20 строк ридов (zcat для архивов)
zcat amp_res_1.fastq.gz | head -n 20

@SRR1363257.37 GWZHISEQ01:153:C1W31ACXX:5:1101:14027:2198 length=101
GGTTGCAGATTCGCAGTGTCGCTGTTCCAGCGCATCACATCTTTGATGTTCACGCCGTGGCGTTTAGCAATGCTTGAAAGCGAATCGCCTTTGCCCACACG
+
@?:=:;DBFADH;CAECEE@@E:FFHGAE4?C?DE<BFGEC>?>FHE4BFFIIFHIBABEECA83;>>@>@CCCDC9@@CC08<@?@BB@9:CC#######
@SRR1363257.46 GWZHISEQ01:153:C1W31ACXX:5:1101:19721:2155 length=101
GTATGAGGTTTTGCTGCATTCTCTGNGCGAATATTAACTCCNTNNNNNTTATAGTTCAAAGCAAGTACCTGTCTCTTATACACATCTCCGAGCCCACGAGC
+
@@<?=D?D==?<AFGDF+AIHEACH#22<:?E8??:9??GG#0#####000;CF=C)4.==CA@@@)=7?C7?E37;3@>;;(.;>AB#############
@SRR1363257.77 GWZHISEQ01:153:C1W31ACXX:5:1101:5069:2307 length=101
GCTTCTCTTAACTGAGGTCACCATCATGCCGTTAAGTCCCTACCTCTCTTTTGCCGGTAACTGTTCCGCCGCGATTGCCTTTTATCTGTCTCTTATACACC
+
??<DBD;4C2=<BB>:AC;<CF<CE@FE9@E1C@891CD*9:?:3D@DD4?D<DD:0;@A=AEIDDA##################################
@SRR1363257.78 GWZHISEQ01:153:C1W31ACXX:5:1101:5178:2440 length=101
GCATAAGGACGATCGCTCCAGAGTAAAATAAATACGCGCATGTGATACTCACAATACCAATGGTGAAGTTACGGGACTTAAACAAACTGAGATCAAGAATC
+
CCCF

Видим стандартный формат FASTQ:  
- 4 строки на каждый рид  
- последовательность нуклеотидов  
- разделитель "+"  
- и строка оценки качества

И делаем вывод, что кодировка Phred33 (в 64 на бывает "?" и "8")

In [27]:
%%bash
cd raw_data

# считаем количество ридов (количество строк/4)
echo "Количество ридов в прямой последовательности:"
zcat amp_res_1.fastq.gz | wc -l | awk '{print $1 / 4}'
echo "Количество ридов в обратной последовательности:"
zcat amp_res_2.fastq.gz | wc -l | awk '{print $1 / 4}'

# количество совпадает, ничего не потеряли.

Количество ридов в прямой последовательности:
455876
Количество ридов в обратной последовательности:
455876


In [25]:
%%bash
cd raw_data

seqkit stats *.fastq.gz

processed files:  0 / 2 [------] ETA: 0s
processed files:  0 / 2 [------] ETA: 0s
processed files:  0 / 2 [------] ETA: 0s
processed files:  0 / 2 [------] ETA: 0s
processed files:  0 / 2 [------] ETA: 0s
processed files:  2 / 2 [======] ETA: 0s
processed files:  2 / 2 [] ETA: 0s. done
processed files:  2 / 2 [] ETA: 0s. done


file                format  type  num_seqs     sum_len  min_len  avg_len  max_len
amp_res_1.fastq.gz  FASTQ   DNA    455,876  46,043,476      101      101      101
amp_res_2.fastq.gz  FASTQ   DNA    455,876  46,043,476      101      101      101


Здесь видим, что длина чтений - строго 101bp

## Шаг 3: оценка качества через FastQC

In [ ]:
%%bash
mkdir fastqc_reports

fastqc -o fastqc_reports/ raw_data/amp_res_1.fastq.gz \
    raw_data/amp_res_2.fastq.gz

Взглянув на отчёты, видим, что Basec Statistics совпадает с расчётами из шага 2: 455876 ридов, длина всех последовательностей по 101bp.

Также видно проблемы:
- **Per base sequence quality**: красный флаг.  
На графике - резкое падение качества после 80bp.  
К 90й позиции медиана уходит в жёлтую зону, а нижнее значение - глубоко в красной.  
Документация fastqc подсказывает, что вероятная причина - деградация химии к концу секвенирования.  
![image.png](https://github.com/bimbomber/HW2_Genomic-Data-Analysis_2026/blob/main/images/image_2026-04-06_15-36-39%20(2).png?raw=true)

- **Per tile sequence quality**.  
Красный флаг на прямой последовательности (Phred score more than 2 less than the mean for that base across all tiles), жёлтый - на обратной (Phred score more than 5 less than the mean for that base across all tiles.).  
На тепловой карте видны полосы и пятна.  
В документации написано, что чаще всего это бывает из-за пылинок, пузырьков воздуха или пятен на проточной ячейке.  
![image-2.png](https://github.com/bimbomber/HW2_Genomic-Data-Analysis_2026/blob/main/images/image_2026-04-06_15-36-39%20(3).png?raw=true)

- **Per base sequence content**: жёлтый флаг.  
Это означает, что разница между A и Т или G и C больше, чем 10% в любой из позиций. 
Это видно на графиках, примерно до 19 позиции количество оснований несбалансировано.  
В документации написано, что это сугубо техническая метрика, которая не может быть скорректирована триммингом, и в большинстве случаев не влияет на результаты дальнейшего анализа. Но вызывает так же Warning в следующем пункте.  
![image-3.png](https://github.com/bimbomber/HW2_Genomic-Data-Analysis_2026/blob/main/images/image_2026-04-06_15-36-39%20(4).png?raw=true)

- **Per sequence GC content**: жёлтый флаг.  
Кривая распределения GC смещена больше чем на 15%, но меньше чем на 30% относительно теоретической кривой для E. coli.  
Как говорилось в прошлом пункте, может быть вызвано небольшими загрязнениями или спецификой подготовки библиотек.  
В целом, распределение выглядит *нормально*, без сильных пиков где-либо. Так что не думаем, что было загрязнение.  
![image-4.png](https://github.com/bimbomber/HW2_Genomic-Data-Analysis_2026/blob/main/images/image_2026-04-06_15-36-39%20(5).png?raw=true)



Логично, что с такими результатами стоит провести тримминг.

## Шаг 4. Фильтрация ридов

In [ ]:
# ● Cut bases off the start of a read if quality below 20
# ● Cut bases off the end of a read if quality below 20
# ● Trim reads using a sliding window approach, with window size 10 and average quality
# within the window 20.
# ● Drop the read if it is below length 20.

# you should check
# the line count (wc -l, divided by 4) of the trimmed paired files manually 

# What happens if we increase the quality score at all steps to 30? 

In [36]:
%%bash
mkdir trimmed_data

# триммим с нужными параметрами
trimmomatic PE -phred33 \
  raw_data/amp_res_1.fastq.gz raw_data/amp_res_2.fastq.gz \
  trimmed_data/amp_res_1_paired.fq.gz trimmed_data/amp_res_1_unpaired.fq.gz \
  trimmed_data/amp_res_2_paired.fq.gz trimmed_data/amp_res_2_unpaired.fq.gz \
  LEADING:20 TRAILING:20 SLIDINGWINDOW:10:20 MINLEN:20

TrimmomaticPE: Started with arguments:
 -phred33 raw_data/amp_res_1.fastq.gz raw_data/amp_res_2.fastq.gz trimmed_data/amp_res_1_paired.fq.gz trimmed_data/amp_res_1_unpaired.fq.gz trimmed_data/amp_res_2_paired.fq.gz trimmed_data/amp_res_2_unpaired.fq.gz LEADING:20 TRAILING:20 SLIDINGWINDOW:10:20 MINLEN:20
Input Read Pairs: 455876 Both Surviving: 446259 (97.89%) Forward Only Surviving: 9216 (2.02%) Reverse Only Surviving: 273 (0.06%) Dropped: 128 (0.03%)
TrimmomaticPE: Completed successfully


- Input Read Pairs: 455876 
- Both Surviving: 446259 (97.89%) 
- Forward Only Surviving: 9216 (2.02%) 
- Reverse Only Surviving: 273 (0.06%) 
- Dropped: 128 (0.03%)

In [38]:
%%bash
# и проверяем вручную, что количество сходится в прямом и обратном прочтении:
cd trimmed_data

echo "Количество ридов в прямой последовательности:"
zcat amp_res_1_paired.fq.gz | wc -l | awk '{print $1 / 4}'
echo "Количество ридов в обратной последовательности:"
zcat amp_res_2_paired.fq.gz | wc -l | awk '{print $1 / 4}'
# всё сходится:

Количество ридов в прямой последовательности:
446259
Количество ридов в обратной последовательности:
446259


In [ ]:
# Проверяем fastqc после тримминга:
!fastqc -o fastqc_reports/ trimmed_data/amp_res_1_paired.fq.gz \
    trimmed_data/amp_res_2_paired.fq.gz

Видим, что **Per base sequence quality** теперь в порядке, обрезка сработала именно так, как мы просили:  
![image.png](https://github.com/bimbomber/HW2_Genomic-Data-Analysis_2026/blob/main/images/image_2026-04-06_15-36-39%20(6).png?raw=true)

И, ожидаемо, **Sequence Length Distribution** стал жёлтым, ведь после обрезки длина не у всех 101bp, как раньше, а minlen=20  
![image-2.png](https://github.com/bimbomber/HW2_Genomic-Data-Analysis_2026/blob/main/images/image_2026-04-06_15-36-39%20(7).png?raw=true)

In [42]:
# Попробуем увеличить везде quality score до 30.
!trimmomatic PE -phred33 \
  raw_data/amp_res_1.fastq.gz raw_data/amp_res_2.fastq.gz \
  trimmed_data/amp_res_1_paired_30.fq.gz trimmed_data/amp_res_1_unpaired_30.fq.gz \
  trimmed_data/amp_res_2_paired_30.fq.gz trimmed_data/amp_res_2_unpaired_30.fq.gz \
  LEADING:30 TRAILING:30 SLIDINGWINDOW:10:30 MINLEN:20

TrimmomaticPE: Started with arguments:
 -phred33 raw_data/amp_res_1.fastq.gz raw_data/amp_res_2.fastq.gz trimmed_data/amp_res_1_paired_30.fq.gz trimmed_data/amp_res_1_unpaired_30.fq.gz trimmed_data/amp_res_2_paired_30.fq.gz trimmed_data/amp_res_2_unpaired_30.fq.gz LEADING:30 TRAILING:30 SLIDINGWINDOW:10:30 MINLEN:20
Input Read Pairs: 455876 Both Surviving: 376340 (82.55%) Forward Only Surviving: 33836 (7.42%) Reverse Only Surviving: 25307 (5.55%) Dropped: 20393 (4.47%)
TrimmomaticPE: Completed successfully


- Input Read Pairs: 455876 
- Both Surviving: 376340 (82.55%) 
- Forward Only Surviving: 33836 (7.42%) 
- Reverse Only Surviving: 25307 (5.55%) 
- Dropped: 20393 (4.47%)

In [ ]:
# И посмотреть на fastqc после такой обрезки
!fastqc -o fastqc_reports/ trimmed_data/amp_res_1_paired_30.fq.gz \
    trimmed_data/amp_res_2_paired_30.fq.gz

Хоть теперь график качества и стал выглядеть лучше, мы потеряли почти 18% ридов в парных чтениях.
![image.png](https://github.com/bimbomber/HW2_Genomic-Data-Analysis_2026/blob/main/images/image_2026-04-06_15-36-39%20(8).png?raw=true)

46mbp (untrimmed) > 42mbp (score20) > 31.3mbp (score30)

Смотря на дальнейший путь анализа, делаем вывод, что повышение порога качества до Q30 делает данные чище, снижая риск принятия ошибок секвенации за мутации, но избыточная фильтрация может привести к потере реальных мутаций из-за снижения покрытия, особенно если мутация находится в плохосеквенированном регионе.

Целесообразно оставить Q20 в качестве баланса между качеством и количеством.

## Шаг 5: выравнивание к референсу

In [ ]:
# Based on the usage details you see, run bwa index on the reference sequence with the default
# options. 

# bwa mem [reference_file] [forward_reads] [reverse_reads] > alignment.sam

# compress and sort the sam file with the commands below. A compressed sam
# file is called a bam file. We can convert a sam file to a bam file as follows: 
# samtools view -S -b alignment.sam > alignment.bam
# Next, run the command below to get some basic statistics.
# samtools flagstat alignment.bam
# What percentage of reads are mapped?

# Sort bam file by sequence coordinate on reference:
# samtools sort alignment.bam alignment_sorted.bam
# (for newer versions of samtools use samtools sort alignment.bam -o alignment_sorted.bam )
# Index bam file for faster search:
# samtools index alignment_sorted.bam

# IGV visualisation

In [ ]:
%%bash
mkdir alignment

# индексируем референс
bwa index raw_data/GCF_000005845.2_ASM584v2_genomic.fna.gz

In [ ]:
# выравнивание ридов и сразу перевод в сжатый BAM для экономии места
!bwa mem raw_data/GCF_000005845.2_ASM584v2_genomic.fna.gz \
  trimmed_data/amp_res_1_paired.fq.gz trimmed_data/amp_res_2_paired.fq.gz | \
  samtools view -S -b > alignment/alignment.bam

In [48]:
# смотрим базовую статистику
!samtools flagstat alignment/alignment.bam

samtools: /home/miracle/miniconda3/bin/../lib/libtinfow.so.6: no version information available (required by samtools)
samtools: /home/miracle/miniconda3/bin/../lib/libncursesw.so.6: no version information available (required by samtools)
samtools: /home/miracle/miniconda3/bin/../lib/libncursesw.so.6: no version information available (required by samtools)
samtools: /home/miracle/miniconda3/bin/../lib/libncursesw.so.6: no version information available (required by samtools)
892776 + 0 in total (QC-passed reads + QC-failed reads)
892518 + 0 primary
0 + 0 secondary
258 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
891649 + 0 mapped (99.87% : N/A)
891391 + 0 primary mapped (99.87% : N/A)
892518 + 0 paired in sequencing
446259 + 0 read1
446259 + 0 read2
888554 + 0 properly paired (99.56% : N/A)
890412 + 0 with itself and mate mapped
979 + 0 singletons (0.11% : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)


Видим следующее:
- Всего 892776 ридов
- Из которых выровнялись к референсу 891649 (99.87%)
- Выровнялись корректно относительно друг друга: 888554 (99.56%)

Делаем вывод о высоком качестве библиотеки и отсутствии значимого загрязнения. Реальный размер фрагментов ДНК соответствует ожиданиям.

In [ ]:
# сортировка и индексация
%%bash
samtools sort alignment/alignment.bam -o alignment/alignment_sorted.bam
samtools index alignment/alignment_sorted.bam

In [ ]:
!samtools flagstat alignment/alignment_sorted.bam
# убедиться, что ничего не потеряли. Напугало изменение размера файла:
# alignment.bam - 92МБ
# alignment_sorted.bam - 71МБ
# Видимо, sort сильнее сжимает.

samtools: /home/miracle/miniconda3/bin/../lib/libtinfow.so.6: no version information available (required by samtools)
samtools: /home/miracle/miniconda3/bin/../lib/libncursesw.so.6: no version information available (required by samtools)
samtools: /home/miracle/miniconda3/bin/../lib/libncursesw.so.6: no version information available (required by samtools)
samtools: /home/miracle/miniconda3/bin/../lib/libncursesw.so.6: no version information available (required by samtools)
892776 + 0 in total (QC-passed reads + QC-failed reads)
892518 + 0 primary
0 + 0 secondary
258 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
891649 + 0 mapped (99.87% : N/A)
891391 + 0 primary mapped (99.87% : N/A)
892518 + 0 paired in sequencing
446259 + 0 read1
446259 + 0 read2
888554 + 0 properly paired (99.56% : N/A)
890412 + 0 with itself and mate mapped
979 + 0 singletons (0.11% : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)


Результаты визуализации в IGV Browser:

![image.png](https://github.com/bimbomber/HW2_Genomic-Data-Analysis_2026/blob/main/images/image_2026-04-06_15-36-39%20(9).png?raw=true)

![image-2.png](https://github.com/bimbomber/HW2_Genomic-Data-Analysis_2026/blob/main/images/image_2026-04-06_15-36-39%20(10).png?raw=true)


## Шаг 6: Поиск генетических вариаций

In [ ]:
# samtools mpileup не хочет брать архивированный файл, поэтому извлекаем
!gunzip -c raw_data/GCF_000005845.2_ASM584v2_genomic.fna.gz > raw_data/ref.fna


In [ ]:
# создаём промежуточный mpileup файл 
!samtools mpileup -f raw_data/ref.fna \
    alignment/alignment_sorted.bam > my.mpileup

In [57]:
# ищем варианты. для примера взяли 80% отличий от референса
!varscan mpileup2snp my.mpileup --min-var-freq 0.8 --variants --output-vcf 1 > VarScan_results.vcf

Only SNPs will be reported
Min coverage:	8
Min reads2:	2
Min var freq:	0.8
Min avg qual:	15
P-value thresh:	0.01
Reading input from my.mpileup
4641343 bases in pileup file
9 variant positions (6 SNP, 3 indel)
0 were failed by the strand-filter
6 variant positions reported (6 SNP, 0 indel)


Видим следующее:
- 4641343 оснований. Просканирован почти весь геном, у нашего штамма [MG1655](https://www.ncbi.nlm.nih.gov/nuccore/U00096.3?report=graph) длина - 4641642
- 9 позиций с вариациями. 
- 6 SNP (полных замен).
- 3 Indels (делеции/инсерции нуклеотидов, которые меняют длину последовательности, но не букву)
- индели не вошли в итоговый отчёт
- 0 ридов отсеяно strand-фильтром. Значит, каждая мутация подтверждена чтениями как с прямой, так и с обратной последовательности. Нет ошибок секвенатора.

In [58]:
# смотрим на результат
!cat VarScan_results.vcf | grep -v "##"

#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	Sample1
NC_000913.3	93043	.	C	G	.	PASS	ADP=17;WT=0;HET=0;HOM=1;NC=0	GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:ADF:ADR	1/1:93:18:17:0:17:100%:4.2852E-10:0:36:0:0:7:10
NC_000913.3	482698	.	T	A	.	PASS	ADP=16;WT=0;HET=0;HOM=1;NC=0	GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:ADF:ADR	1/1:87:16:16:0:16:100%:1.6637E-9:0:45:0:0:7:9
NC_000913.3	852762	.	A	G	.	PASS	ADP=14;WT=0;HET=0;HOM=1;NC=0	GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:ADF:ADR	1/1:76:14:14:0:14:100%:2.4927E-8:0:36:0:0:8:6
NC_000913.3	1905761	.	G	A	.	PASS	ADP=13;WT=0;HET=0;HOM=1;NC=0	GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:ADF:ADR	1/1:70:13:13:0:13:100%:9.6148E-8:0:44:0:0:11:2
NC_000913.3	3535147	.	A	C	.	PASS	ADP=17;WT=0;HET=0;HOM=1;NC=0	GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:ADF:ADR	1/1:93:17:17:0:17:100%:4.2852E-10:0:36:0:0:10:7
NC_000913.3	4390754	.	G	T	.	PASS	ADP=15;WT=0;HET=0;HOM=1;NC=0	GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:ADF:ADR	1/1:81:16:15:0:15:100%:6.

- FREQ(частоота)=100%; GT(генотип)=1/1; RD(reference reads)=0. Во всех ридах в этой позиции референсный нуклеотид заменён на альтернативный.
- GQ(genotype quality) = 70-93. Высокий показатель.

Отличия от референса есть явные. Это не шумы.


In [ ]:
# Explore all the mutations, 
# find whether each mutation occurs in a gene, 
# whether it is missense (changes the amino acid sequence), 
# nonsense (introduces a frameshift or early stop codon), or
# synonymous (no amino acid change). 
# For missense or nonsense mutations find out what that gene name is.

Визуализируем в IGV.

93043:  
![image.png](https://github.com/bimbomber/HW2_Genomic-Data-Analysis_2026/blob/main/images/image_2026-04-06_15-36-39.png?raw=true)  
Ген ftsI;    
missense (GCC > GGC; A > G)  

482698:  
![image-7.png](https://github.com/bimbomber/HW2_Genomic-Data-Analysis_2026/blob/main/images/image_2026-04-06_15-36-40.png?raw=true)  
Ген acrB;  
missense (CAG > CTG; Q > L)  

852762:  
![image-3.png](https://github.com/bimbomber/HW2_Genomic-Data-Analysis_2026/blob/main/images/image_2026-04-06_15-36-40%20(2).png?raw=true)  
intragenic_variant. Мутация вне кодирующей части.

1905761:  
![image-4.png](https://github.com/bimbomber/HW2_Genomic-Data-Analysis_2026/blob/main/images/image_2026-04-06_15-36-40%20(3).png?raw=true)  
Ген mntP;  
missense (GGT > GAT; G > D)  

3535147:  
![image-5.png](https://github.com/bimbomber/HW2_Genomic-Data-Analysis_2026/blob/main/images/image_2026-04-06_15-36-40%20(5).png?raw=true)  
Ген envZ;  
missense (GTA > GGA; V > G)  

4390754:  
![image-2.png](https://github.com/bimbomber/HW2_Genomic-Data-Analysis_2026/blob/main/images/image_2026-04-06_15-36-40%20(4).png?raw=true)  
Ген rsgA;  
synonymous (GCC > GCA; A > A)  

## Шаг 8: автоматическая аннотация с SNP

In [ ]:
%%bash
# создаём директорию для этого шага
mkdir snp_annot
cd snp_annot

# записываем конфиг
echo "k12.genome : ecoli_K12" > snpEff.config

# создаём директорию для базы данных и кладём туда gbk 
mkdir -p data/k12
wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/005/845/GCF_000005845.2_ASM584v2/GCF_000005845.2_ASM584v2_genomic.gbff.gz

gunzip -c GCF_000005845.2_ASM584v2_genomic.gbff.gz > data/k12/genes.gbk

In [ ]:
%%bash
cd snp_annot

# строим БД
snpEff build -genbank -v k12

In [66]:
%%bash
cd snp_annot

# аннотируем
snpEff ann k12 ../VarScan_results.vcf > VarScan_results_annotated.vcf

In [67]:
# смотрим на результат
!cat snp_annot/VarScan_results_annotated.vcf | grep -v "##"

#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	Sample1
NC_000913.3	93043	.	C	G	.	PASS	ADP=17;WT=0;HET=0;HOM=1;NC=0;ANN=G|missense_variant|MODERATE|ftsI|b0084|transcript|b0084|protein_coding|1/1|c.1631C>G|p.Ala544Gly|1631/1767|1631/1767|544/588||,G|upstream_gene_variant|MODIFIER|murE|b0085|transcript|b0085|protein_coding||c.-123C>G|||||123|WARNING_TRANSCRIPT_NO_START_CODON,G|upstream_gene_variant|MODIFIER|murF|b0086|transcript|b0086|protein_coding||c.-1607C>G|||||1607|,G|upstream_gene_variant|MODIFIER|mraY|b0087|transcript|b0087|protein_coding||c.-2959C>G|||||2959|,G|upstream_gene_variant|MODIFIER|murD|b0088|transcript|b0088|protein_coding||c.-4044C>G|||||4044|,G|downstream_gene_variant|MODIFIER|cra|b0080|transcript|b0080|protein_coding||c.*4011C>G|||||4011|WARNING_TRANSCRIPT_NO_START_CODON,G|downstream_gene_variant|MODIFIER|mraZ|b0081|transcript|b0081|protein_coding||c.*2951C>G|||||2951|,G|downstream_gene_variant|MODIFIER|rsmH|b0082|transcript|b0082|protein_coding||c.*2008C>G|||||2008|,

Аннотация подтверждает наши визуальные наблюдения. И она куда удобнее и быстрее.